# Séance 3 : nettoyer

Objectif : partir d'un fichier brut, tel qu'il arrive du terrain, et obtenir un tableau propre. Puis **prouver** qu'il est propre en le comparant au fichier de référence.

Règle du jour : on compte avant et après chaque étape (`shape`, `isna().sum()`, `duplicated().sum()`), on ne supprime jamais à l'aveugle.

## 1. Lire et diagnostiquer

In [1]:
import pandas as pd

brut = pd.read_csv("donnees/prix_marches_brut.csv")
brut.shape

(2344, 6)

In [2]:
brut.dtypes

mois           str
departement    str
marche         str
produit        str
unite          str
prix_htg       str
dtype: object

`prix_htg` est lu comme du texte (`str`) : au moins une valeur de la colonne n'est pas un nombre. Tant que c'est du texte, impossible de calculer une moyenne.

In [3]:
brut.isna().sum()

mois            0
departement     0
marche          0
produit         0
unite           0
prix_htg       70
dtype: int64

In [4]:
brut.duplicated().sum()

np.int64(40)

In [5]:
brut["departement"].unique()

<StringArray>
[ 'Artibonite',      'Centre',  'Grand'Anse',      'Nippes',        'Nord',
    'Nord Est',    'Nord-Est',  'Nord-Ouest',       'Ouest',         'Sud',
     'Sud-Est',  'artibonite',      'nippes',     'Sud Est',      'centre',
         'sud', 'Artibonite ',       'OUEST',  'Grand Anse', 'Grande-Anse',
  'Nord Ouest',        'NORD',       'ouest',     'sud-est',      ' Ouest',
  'nord-ouest']
Length: 26, dtype: str

In [6]:
brut["produit"].unique()

<StringArray>
[    'Riz importé',       'Riz local',      'Maïs moulu',   'Farine de blé',
    'Haricot noir',  'Huile végétale',           'Sucre',       'Spaghetti',
  'farine de blé ',   'haricot noir ',    'riz importé ',          'sucre ',
      'riz local ',      'spaghetti ',     'maïs moulu ', 'huile végétale ']
Length: 16, dtype: str

Le diagnostic tient en cinq lignes : `shape`, `dtypes`, `isna().sum()`, `duplicated().sum()`, et `unique()` sur chaque colonne de texte. On note les défauts trouvés avant de toucher au tableau :

| Défaut | Où |
|---|---|
| prix en texte, 70 prix manquants | `prix_htg` |
| 40 lignes en double | tout le tableau |
| 26 graphies pour 10 départements | `departement` |
| minuscules et espace final | `produit` |

## 2. Les prix : du texte aux nombres

`pd.to_numeric(..., errors="coerce")` convertit ce qu'elle peut et met `NaN` (manquant) sur le reste. Essayons directement :

In [7]:
pd.to_numeric(brut["prix_htg"], errors="coerce").isna().sum()

np.int64(150)

150 manquants, alors que le fichier n'en avait que 70 : la conversion a jeté 80 valeurs qui étaient des nombres mal écrits. Regardons lesquelles.

In [8]:
texte = brut["prix_htg"]
texte[pd.to_numeric(texte, errors="coerce").isna() & texte.notna()].head()

16       700,0
17     343 HTG
180      332,0
186    559 HTG
192     81 HTG
Name: prix_htg, dtype: str

Deux défauts : un suffixe « HTG » et une virgule décimale. On répare le texte **avant** de convertir, sur une copie du tableau.

In [9]:
propre = brut.copy()
propre["prix_htg"] = propre["prix_htg"].str.replace(" HTG", "").str.replace(",", ".")
propre["prix_htg"] = pd.to_numeric(propre["prix_htg"], errors="coerce")
propre["prix_htg"].isna().sum()

np.int64(70)

In [10]:
propre["prix_htg"].dtype

dtype('float64')

## 3. Le texte : `strip`, `title`, `replace`

Les méthodes de chaînes s'appliquent à toute une colonne avec `.str` : `.str.strip()` retire les espaces autour, `.str.title()` met une majuscule à chaque mot.

In [11]:
propre["departement"] = propre["departement"].str.strip().str.title()
propre["departement"].unique()

<StringArray>
[ 'Artibonite',      'Centre',  'Grand'Anse',      'Nippes',        'Nord',
    'Nord Est',    'Nord-Est',  'Nord-Ouest',       'Ouest',         'Sud',
     'Sud-Est',     'Sud Est',  'Grand Anse', 'Grande-Anse',  'Nord Ouest']
Length: 15, dtype: str

Il reste des graphies différentes pour un même département. Pour ça, un **dictionnaire de correspondance** et `replace` :

In [12]:
correspondance = {"Grande-Anse": "Grand'Anse", "Grand Anse": "Grand'Anse", "Nord Ouest": "Nord-Ouest",
                  "Nord Est": "Nord-Est", "Sud Est": "Sud-Est"}
propre["departement"] = propre["departement"].replace(correspondance)
sorted(propre["departement"].unique())

['Artibonite',
 'Centre',
 "Grand'Anse",
 'Nippes',
 'Nord',
 'Nord-Est',
 'Nord-Ouest',
 'Ouest',
 'Sud',
 'Sud-Est']

Pour les produits, `title()` donnerait « Riz Importé » : on veut une majuscule au premier mot seulement, c'est `capitalize()`.

In [13]:
propre["produit"] = propre["produit"].str.strip().str.capitalize()
propre["produit"].unique()

<StringArray>
[   'Riz importé',      'Riz local',     'Maïs moulu',  'Farine de blé',
   'Haricot noir', 'Huile végétale',          'Sucre',      'Spaghetti']
Length: 8, dtype: str

## 4. Les doublons

In [14]:
propre.duplicated().sum()

np.int64(40)

In [15]:
propre[propre.duplicated(keep=False)].sort_values(["mois", "marche", "produit"]).head(4)

,mois,departement,marche,produit,unite,prix_htg
54,2024-01,Nord-Ouest,Port-de-Paix,Riz importé,marmite,547.0
62,2024-01,Nord-Ouest,Port-de-Paix,Riz importé,marmite,547.0
142,2024-02,Nord,Cap-Haïtien,Sucre,marmite,467.0
144,2024-02,Nord,Cap-Haïtien,Sucre,marmite,467.0


`duplicated()` marque la deuxième occurrence ; `keep=False` les marque toutes, pour les voir par paires. `drop_duplicates()` garde la première.

In [16]:
propre = propre.drop_duplicates()
propre.shape

(2304, 6)

## 5. Les valeurs manquantes : supprimer ou remplir

Deux choix, et aucun n'est neutre : `dropna` perd des lignes, `fillna` invente une valeur. Regardons l'effet sur la moyenne.

In [17]:
propre[propre["prix_htg"].isna()].head(3)

,mois,departement,marche,produit,unite,prix_htg
42,2024-01,Nord,Cap-Haïtien,Haricot noir,marmite,NaN
155,2024-02,Nord-Ouest,Port-de-Paix,Riz local,marmite,NaN
184,2024-02,Sud,Les Cayes,Spaghetti,paquet,NaN


In [18]:
print("moyenne, NaN ignorés :", round(propre["prix_htg"].mean(), 1))
print("moyenne avec fillna(0) :", round(propre["prix_htg"].fillna(0).mean(), 1))
print("moyenne avec la médiane :", round(propre["prix_htg"].fillna(propre["prix_htg"].median()).mean(), 1))

moyenne, NaN ignorés : 804.0
moyenne avec fillna(0) : 780.0
moyenne avec la médiane : 799.9


`mean` ignore déjà les `NaN`. Remplir par 0 fausse tout : un prix de 0 gourde n'existe pas. Ici on n'a aucun moyen de retrouver les 70 prix, donc on supprime ces lignes, **en le disant** avec `subset`.

In [19]:
propre = propre.dropna(subset=["prix_htg"])
propre.shape

(2235, 6)

Sans `subset`, `dropna()` supprime toute ligne qui a un manque n'importe où : c'est souvent trop.

## 6. Les valeurs aberrantes

Un tableau sans `NaN` ni doublon peut encore contenir des valeurs fausses. `describe` et `nlargest` les font ressortir.

In [20]:
propre["prix_htg"].describe().round(1)

count     2235.0
mean       804.0
std        710.1
min         69.0
25%        481.5
50%        666.0
75%        941.5
max      16660.0
Name: prix_htg, dtype: float64

In [21]:
propre.nlargest(5, "prix_htg")[["mois", "marche", "produit", "prix_htg"]]

,mois,marche,produit,prix_htg
256,2024-03,Port-de-Paix,Huile végétale,16660.0
1353,2025-02,Jacmel,Haricot noir,10560.0
1007,2024-11,Miragoâne,Haricot noir,9720.0
1003,2024-11,Miragoâne,Riz importé,6870.0
538,2024-06,Fort-Liberté,Farine de blé,5060.0


Un gallon d'huile à 16 660 gourdes : une virgule décalée à la saisie, le prix est 10 fois trop grand. Règle simple : un prix qui dépasse **3 fois la médiane de son produit** est suspect. On corrige produit par produit, avec `loc`.

In [22]:
for produit in propre["produit"].unique():
    du_produit = propre["produit"] == produit
    mediane = propre.loc[du_produit, "prix_htg"].median()
    suspect = du_produit & (propre["prix_htg"] > 3 * mediane)
    if suspect.sum() > 0:
        print(produit, ":", int(suspect.sum()), "prix divisé(s) par 10")
    propre.loc[suspect, "prix_htg"] = propre.loc[suspect, "prix_htg"] / 10
propre["prix_htg"].max()

Riz importé : 1 prix divisé(s) par 10
Maïs moulu : 1 prix divisé(s) par 10
Farine de blé : 1 prix divisé(s) par 10
Haricot noir : 2 prix divisé(s) par 10
Huile végétale : 1 prix divisé(s) par 10
Spaghetti : 1 prix divisé(s) par 10


np.float64(2884.0)

## 7. Les dates

`mois` est du texte : « 2024-01 ». Ça se trie bien, mais on ne peut ni extraire l'année ni calculer un écart. `pd.to_datetime` en fait une vraie date, et `.dt` donne accès à ses morceaux.

In [23]:
propre["date"] = pd.to_datetime(propre["mois"])
propre[["mois", "date"]].head(3)

,mois,date
0,2024-01,2024-01-01
1,2024-01,2024-01-01
2,2024-01,2024-01-01


In [24]:
propre["date"].dt.year.unique()

array([2024, 2025], dtype=int32)

In [25]:
propre[propre["date"].dt.month == 12].shape

(189, 7)

## 8. La preuve : comparer au fichier de référence

Les 2 304 relevés propres existent dans `prix_marches.csv`. Notre tableau doit lui être identique, aux 70 prix manquants près. On construit une clé par relevé (mois, département, marché, produit) pour aligner les deux tableaux.

In [26]:
officiel = pd.read_csv("donnees/prix_marches.csv")
officiel.shape, propre.shape

((2304, 6), (2235, 7))

In [27]:
cle = ["mois", "departement", "marche", "produit"]
propre = propre.sort_values(cle).reset_index(drop=True)
officiel = officiel.sort_values(cle).reset_index(drop=True)

cle_propre = propre["mois"] + "|" + propre["departement"] + "|" + propre["marche"] + "|" + propre["produit"]
cle_officiel = officiel["mois"] + "|" + officiel["departement"] + "|" + officiel["marche"] + "|" + officiel["produit"]
attendu = officiel[cle_officiel.isin(cle_propre)].reset_index(drop=True)
attendu.shape

(2235, 6)

In [28]:
propre[attendu.columns].equals(attendu)

True

In [29]:
propre[attendu.columns].compare(attendu)

Empty DataFrame
Columns: []
Index: []

`equals` répond vrai ou faux ; `compare` montre les cases qui diffèrent, et un tableau vide quand tout est identique.

## Exercices en binôme

Les énoncés et les résultats attendus sont sur la page de la séance 3.

## Aller plus loin

- **La médiane du groupe** : `propre.groupby("produit")["prix_htg"].transform("median")` donne, pour chaque ligne, la médiane de son produit. C'est la question Aller plus loin de l'issue 3 du TP1.
- **Les expressions régulières** : `brut["prix_htg"].str.extract(r"(\d+)")` récupère les chiffres d'un texte, quelle que soit la façon dont le reste est écrit.
- **L'écart interquartile** : une valeur au-delà de Q3 + 1,5 × (Q3 - Q1) est aberrante. Comparez cette règle à « 3 fois la médiane ».

## Sources

- pandas, « Working with missing data » : https://pandas.pydata.org/docs/user_guide/missing_data.html
- pandas, « Working with text data » : https://pandas.pydata.org/docs/user_guide/text.html
- pandas, « Time series / date functionality » : https://pandas.pydata.org/docs/user_guide/timeseries.html
- Wes McKinney, *Python for Data Analysis*, chapitre 7, « Data Cleaning and Preparation » : https://wesmckinney.com/book/data-cleaning